In [2]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from array import array
import os
from PIL import Image
import sys
import time

In [3]:
from dotenv import load_dotenv
load_dotenv()

subscription_key = os.getenv("VISION_KEY")
endpoint = os.getenv("VISION_ENDPOINT")
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))

In [4]:
# img = open("data/test1.png", "rb")
img = open("data/test2.jpeg", "rb")
read_response = computervision_client.read_in_stream(
    image=img,
    mode="Printed",
    raw=True
)

operation_id = read_response.headers['Operation-Location'].split('/')[-1]
while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

result = []
ocr_boxes = []
if read_result.status == OperationStatusCodes.succeeded:
    for text_result in read_result.analyze_result.read_results:
        for line in text_result.lines:
            print(line.text)
            result.append(line.text)
            ocr_boxes.append(line.bounding_box)

print()

Lucces in resolvarea
TEMELOR la
LABORA toarele de
Inteligenta Artificialà!



In [5]:
def levenshtein_distance(s1, s2):
    m, n = len(s1), len(s2)
    
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(m + 1):
        dp[i][0] = i
    
    for j in range(n + 1):
        dp[0][j] = j
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],   
                    dp[i][j-1],    
                    dp[i-1][j-1]                   
                )
    
    return dp[m][n]

def cer(ground_truth, hypothesis):
    gt  = ground_truth.replace(" ", "")
    hyp = hypothesis.replace(" ", "")
    
    dist = levenshtein_distance(gt, hyp)
    rate = dist / max(len(gt), 1)   
    return dist, rate

def wer(ground_truth, hypothesis):
    gt_words  = ground_truth.lower().split()
    hyp_words = hypothesis.lower().split()
    
    dist = levenshtein_distance(gt_words, hyp_words)
    rate = dist / max(len(gt_words), 1)
    return dist, rate

def hamming_distance(s1, s2):
    max_len = max(len(s1), len(s2))
    s1_pad = s1.ljust(max_len)
    s2_pad = s2.ljust(max_len)
    dist = sum(c1 != c2 for c1, c2 in zip(s1_pad, s2_pad))
    return dist, dist / max_len

def jaro_winkler_approx(s1, s2):
    _, cer_rate = cer(s1, s2)
    return 1 - cer_rate 

def evaluate_all_metrics(gt, ocr):
    lev_dist, cer_rate = cer(gt, ocr)
    wer_dist, wer_rate = wer(gt, ocr)
    ham_dist, ham_rate = hamming_distance(gt, ocr)
    jw_approx = jaro_winkler_approx(gt, ocr)
    return {
        "Levenshtein": lev_dist,
        "CER": round(cer_rate, 3),
        "WER": round(wer_rate, 3),
        "Hamming": f"{ham_dist} ({ham_rate:.3f})",
        "Jaro-Winkler-approx": round(jw_approx, 3)
    }

ground_truth = "Succes în rezolvarea tEMELOR la LABORAtoarele de Inteligență Artificială!"
detected_text = " ".join(result)

metrics = evaluate_all_metrics(ground_truth, detected_text)
for metric, val in metrics.items():
    print(f"{metric}: {val}")

Levenshtein: 7
CER: 0.108
WER: 0.778
Hamming: 40 (0.541)
Jaro-Winkler-approx: 0.892


In [6]:

def box_to_coords(box):
    xs = box[0::2]
    ys = box[1::2]
    return min(xs), min(ys), max(xs), max(ys)

def iou(box1, box2):
    x1_min, y1_min, x1_max, y1_max = box_to_coords(box1)
    x2_min, y2_min, x2_max, y2_max = box_to_coords(box2)

    xi1 = max(x1_min, x2_min)
    yi1 = max(y1_min, y2_min)
    xi2 = min(x1_max, x2_max)
    yi2 = min(y1_max, y2_max)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    union_area = box1_area + box2_area - inter_area

    return inter_area / union_area if union_area != 0 else 0

gt_boxes = [
    [81, 302, 1327, 462],
    [130, 581, 1047, 723],
    [82, 922, 999, 1024],
    [107, 1128, 1451, 1369]
]

for i, (ocr_box, gt_box) in enumerate(zip(ocr_boxes, gt_boxes)):
    iou_val = iou(ocr_box, gt_box)
    correct = iou_val > 0.5
    print(f"Linia {i+1}: IoU={iou_val:.2f}, Localizare corectă={correct}")

num_correct = sum(iou(ocr_box, gt_box) > 0.5 for ocr_box, gt_box in zip(ocr_boxes, gt_boxes))
accuracy = num_correct / len(gt_boxes)
print(f"Acuratețe localizare: {accuracy*100:.2f}%")

Linia 1: IoU=0.83, Localizare corectă=True
Linia 2: IoU=0.92, Localizare corectă=True
Linia 3: IoU=0.81, Localizare corectă=True
Linia 4: IoU=0.68, Localizare corectă=True
Acuratețe localizare: 100.00%


In [7]:
# Cresterea contrastului si luminozitatii
# Corectarea orientarii (sa fie cat mai drept)
# Filtrare zgomot (ex: blur, pentru a elimina petele sau pixelii izolati)
# Segmentarea imaginilor (crop pe partile cu text)
# Testarea mai multor modele

In [11]:
from PIL import Image, ImageEnhance

img = Image.open("data/test2.jpeg")
img = ImageEnhance.Brightness(img).enhance(1.3)
img = ImageEnhance.Contrast(img).enhance(1.5)
img.save("data/test2_enhanced.jpeg")
img_file = open("data/test2_enhanced.jpeg", "rb")

read_response = computervision_client.read_in_stream(
    image=img_file,
    mode="Printed",
    raw=True
)

operation_id = read_response.headers['Operation-Location'].split('/')[-1]
while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

result = []
ocr_boxes = []
if read_result.status == OperationStatusCodes.succeeded:
    for text_result in read_result.analyze_result.read_results:
        for line in text_result.lines:
            print(line.text)
            result.append(line.text)
            ocr_boxes.append(line.bounding_box)

print()

Lucces in resolvarea
TEMELOR la
LABORA toarele de
Inteligenta Artificialà!

